In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [2]:
import logging
import delta_sharing
import pandas as pd
from datetime import datetime

import general_functions.databricks_client as db_client
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import send_to_innkeepr_api_paginated

In [3]:
customer = "Pendix"
path_data = f"SprintStories/Project-Intervention-Logging/data/"
url = return_api_url()
print(f"url = {url}")
workspace_id_lists  = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_id_lists if acc["name"] == customer]
try:
    workspace_id = workspace_id[0]
except:
    print("could not be found")
    names = [acc["name"] for acc in workspace_id_lists]
    names = sorted(names)
    print(names)

In [4]:
profile_path = db_client.return_databricks_client()
table_name = "incident_detections"
table_path = f"{profile_path}#delta_share_events.incident_monitoring.{table_name}"
incident_detection = delta_sharing.load_as_pandas(table_path)# , limit=100000)
incident_detection

In [5]:
incident_detection.to_csv(f"{path_data}_incident_detection.csv")

In [6]:
incident_detection.columns

In [7]:
incident_detection["detected_date"].min()

In [8]:
incident_detection[incident_detection["signals"].astype("str").str.contains("693c077ca5250a665da36c80")]

In [9]:
incident_detection[(incident_detection["workspace_id"]=="6870b934768354324d58e9cf")&(incident_detection["connection"]=="facebook")]["signals"].astype("str").value_counts()

In [10]:
incident_detection["signals"].astype("str").value_counts()

In [11]:
incident_detection_explode = incident_detection.explode("signals")
incident_detection_explode = incident_detection_explode[incident_detection_explode["signals"].isnull()==False]
incident_detection_explode["signal_id"] = incident_detection_explode["signals"].apply(lambda x: x["signal_id"])
incident_detection_explode.head()

In [12]:
# check for missing signals
interventions = pd.read_csv("SprintStories/Project-Intervention-Logging/data/_intervention_logging_dev_2026-09-07 11:22:44.008563_reduced.csv")
interventions = interventions[interventions["created_at"]>="2026-08-20"]
print(f"filtered interventions date range: {interventions['created_at'].min()} to {interventions['created_at'].max()}")
interventions = interventions.sort_values(by="created_at", ascending=False)
interventions_wo_baseline = interventions[interventions["baseline_cpa"].isnull()]
interventions_wo_baseline.head()

In [13]:
interventions_wo_detections = interventions_wo_baseline[~interventions_wo_baseline["intervention_id"].isin(incident_detection_explode["signal_id"])][["workspace_id", "workspace_name", "signal_id"]].drop_duplicates()
interventions_wo_detections